In [1]:
from datasets import load_dataset
from helpers import load_qwen_model, get_clip_generator, run_inference, visualize_clip
import time
import torch

dataset = load_dataset("flwrlabs/ucf101")

# To see what's inside (columns, number of rows, etc.)
print(dataset)

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/79 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'video_id', 'clip_id', 'frame', 'label'],
        num_rows: 1786096
    })
    test: Dataset({
        features: ['image', 'video_id', 'clip_id', 'frame', 'label'],
        num_rows: 697222
    })
})


In [3]:
test_data = dataset["test"]

#press q to quit the video window
for clip in get_clip_generator(test_data, max_rows=10):
    print(f"\n🎬 Clip: {clip['clip_id']} | Expected: {clip['label_name']}")
    
    visualize_clip(clip['images'], fps = 30)  # Optional: visualize the clip frames 


🎬 Clip: v_ApplyEyeMakeup_g01_c01 | Expected: ApplyEyeMakeup
Press 'q' to close the video window.


In [ ]:

# Initialize tracking variables
total_processing_time = 0.0
total_frames_processed = 0.0

# Initialize Model
model, processor = load_qwen_model()
test_data = dataset["test"]

print("🚀 Starting Pipeline...")

for clip in get_clip_generator(test_data, max_rows=1000):
    print(f"\n🎬 Clip: {clip['clip_id']} | Expected: {clip['label_name']}")
    
    # --- Start Timer ---
    start_time = time.time()
    
    # Run inference
    response = run_inference(model, processor, clip['images'])
    
    # --- End Timer ---
    end_time = time.time()
    
    # Calculate stats for this specific clip
    # Note: We use the length of the downsized frames for the 'per frame' calculation
    # since those are what the model actually "saw"
    num_frames = len(clip['images']) # Or use len(downsized) inside run_inference if preferred
    duration = end_time - start_time
    
    total_processing_time += duration
    total_frames_processed += num_frames
    
    avg_time_per_frame = duration / num_frames if num_frames > 0 else 0
    
    print(f"🤖 Qwen: {response}")
    print(f"⏱️  Clip Time: {duration:.2f}s | Avg Time per Frame: {avg_time_per_frame:.4f}s")
    
    
    
    # Clear memory
    torch.cuda.empty_cache()

# Global average update
global_avg = total_processing_time / total_frames_processed
print(f"📊 Running Global Average: {global_avg:.4f}s per frame")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

🚀 Starting Pipeline...

🎬 Clip: v_ApplyEyeMakeup_g01_c01 | Expected: ApplyEyeMakeup
🤖 Qwen: A woman is applying makeup to her eyes with a brush. She is looking down at her face and moving the brush back and forth across her eyelids.
⏱️  Clip Time: 3.37s | Avg Time per Frame: 0.0205s

🎬 Clip: v_ApplyEyeMakeup_g01_c02 | Expected: ApplyEyeMakeup
🤖 Qwen: A woman is applying makeup to her eyes with a brush. She is looking down at her face and moving the brush across her eyelids.
⏱️  Clip Time: 2.24s | Avg Time per Frame: 0.0182s

🎬 Clip: v_ApplyEyeMakeup_g01_c03 | Expected: ApplyEyeMakeup
🤖 Qwen: A woman is applying makeup to her eyes with a brush. She is looking directly at the camera and appears to be demonstrating how to apply the product.
⏱️  Clip Time: 2.90s | Avg Time per Frame: 0.0112s

🎬 Clip: v_ApplyEyeMakeup_g01_c04 | Expected: ApplyEyeMakeup
🤖 Qwen: A woman is applying makeup to her eyes with a brush. She is looking directly at the camera and smiling.
⏱️  Clip Time: 2.27s | Avg T